**Welcome to the Mini Challenge!**

In this notebook, your task is to explore and report on the decision-making process of a simple CNN model trained on an image classification task. The model, trained on a varient of the MNIST dataset (a 10-class classification problem), will be loaded below along with 10 example images.

Your goal is to apply various Explainable AI (XAI) techniques to understand how the model makes decisions. Keep in mind that some XAI methods are data-agnostic. Just because you learned them in a different context doesn't mean they can't be applied to image data.

For details on grading, please refer to the "Proof of Performance" section in the EAI space.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings # to suppress warnings
warnings.filterwarnings("ignore") # ignore warnings

In [ ]:
# Define the CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
# Load the models best weights
model = SimpleCNN()
model.load_state_dict(torch.load('../models/challenge_model.pth'))
model.eval()

In [ ]:
# Load data and labels
challenge_images = np.load('../data/challenge_images.npy')
challenge_labels = np.load('../data/challenge_labels.npy')
print(f"Loaded challenge images: {challenge_images.shape}, labels: {challenge_labels.shape}")

# EAI Mini Challenge

This MC applies three XAI methods (Grad-CAM, Integrated Gradients and Occlusion) to find out, what a simple CNN that was trained on a variant of the MNIST dataset has actually learned.

## Data Overview

In [ ]:
import torch.nn.functional as F

# Convert images to tensor
images_tensor = torch.tensor(challenge_images, dtype=torch.float32)

# Compute predictions
with torch.no_grad():
    outputs = model(images_tensor)
    predictions = torch.argmax(outputs, dim=1).numpy()

# Visualize images with labels and predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(challenge_images[i, 0], cmap='gray')
    ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

The model predicts all 10 images correctly according to their labels. However, the labels do not match the digits shown in the images. For example, an image showing a 0 carries the label 1 and an image showing a 5 is labeled 3. This suggests the model may not have learned to recognize the digit shapes themselves, but rather some other signal in the data that correlates with the (incorrect) labels.

## Grad-CAM

Grad-CAM (Gradient weighted Class Activation Mapping) highlights the regions of an image that most influenced the model's prediction. It does this by computing the gradients of the predicted class score with respect to the last convolutional layer's feature maps. Regions with strong positive gradients are here considered most relevant for the decision.

In [ ]:
from captum.attr import LayerGradCam, LayerAttribution
from scipy.ndimage import gaussian_filter

# Initialize Grad-CAM on last conv layer
gradcam = LayerGradCam(model, model.conv2)

def compute_gradcam(img_np, target, smooth=False):
    img = torch.tensor(img_np, dtype=torch.float32)
    attr = gradcam.attribute(img, target=target)
    # Upsample to original image size
    heatmap = LayerAttribution.interpolate(attr, (28, 28))[0, 0].detach().numpy()
    # Keep only positive attributions and normalize
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    if smooth:
        heatmap = gaussian_filter(heatmap, sigma=1)
    return heatmap

def plot_gradcam(smooth=False):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        heatmap = compute_gradcam(challenge_images[i:i+1], int(predictions[i]), smooth=smooth)
        ax.imshow(challenge_images[i, 0], cmap='gray')
        ax.imshow(heatmap, cmap='jet', alpha=0.4)
        ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
        ax.axis('off')
    plt.suptitle("Grad-CAM (smoothed)" if smooth else "Grad-CAM")
    plt.tight_layout()
    plt.show()

plot_gradcam(smooth=False)

Grad-CAM reveals that the model's attention is not consistently focused on the digit itself. For several images, notable activations appear in the corners, suggesting the model may be picking up on features outside the digit region. The pattern varies across images, which makes it hard to make confident conclusions at this point.

The same attributions are shown below with a Gaussian smoothing applied to the heatmap for cleaner visualization.

In [ ]:
plot_gradcam(smooth=True)

The smoothed version makes the spatial structure a bit clearer and shows that the corner activations seem to be more consistent than they first appeared.

## Integrated Gradients

Integrated Gradients attributes the model's prediction to each input pixel by computing the average gradient along a straight path from a baseline (a black image) to the actual input. Unlike Grad-CAM, it operates directly in pixelspace and does not depend on a specific layer, which makes it a more detailed attribution method. Given the Grad-CAM results, it seems worth investigating whether the corner activations show up here as well.

In [ ]:
from captum.attr import IntegratedGradients

ig = IntegratedGradients(model)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = torch.tensor(challenge_images[i:i+1], dtype=torch.float32, requires_grad=True)
    target = int(predictions[i])
    
    # Compute attributions with black image as baseline
    attr = ig.attribute(img, target=target, baselines=torch.zeros_like(img))
    heatmap = attr[0, 0].detach().numpy()
    
    # Take absolute value and normalize
    heatmap = np.abs(heatmap)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    
    ax.imshow(challenge_images[i, 0], cmap='gray')
    ax.imshow(heatmap, cmap='jet', alpha=0.4)
    ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
    ax.axis('off')

plt.suptitle("Integrated Gradients")
plt.tight_layout()
plt.show()

Integrated Gradients produces a much more pixel-precise picture than Grad-CAM. Across almost all images, the strongest attributions appear along the top (left) edge of the image rather than on the digit itself, which suggests the model may be relying on something in that region to make its predictions. The digit shapes contribute very little to not at all to the attributions, which seems to support the suspicion raised by Grad-CAM. What pattern exactly the model is picking up on in that area remains not fully clear.